# RAG From Scratch: Hybrid Search and Reranking

Dense retrieval is not always enough. Modern RAG systems often combine sparse lexical retrieval, dense semantic retrieval, reciprocal rank fusion, and reranking.

In [ ]:
import sys
!uv pip install --python {sys.executable} -r pyproject.toml

## Model provider

Shared provider selector for notebook generation cells. `RAG_CHAT_PROVIDER=gemini` keeps the original Gemini chat path. Set `RAG_CHAT_PROVIDER=openrouter` and `OPENROUTER_API_KEY` in `.env` to use OpenRouter cheap models. Retrieval examples default to `RAG_EMBEDDING_PROVIDER=local` for API-free TF-IDF embeddings; set `RAG_EMBEDDING_PROVIDER=gemini` for the original Gemini embeddings.


In [ ]:
from rag_providers import (
    OPENROUTER_MODELS,
    chat_model,
    chat_provider_name,
    embedding_model,
    embedding_provider_name,
    openrouter_model_name,
)

print("Chat provider:", chat_provider_name())
print("Embedding provider:", embedding_provider_name())
print("OpenRouter model:", openrouter_model_name())
OPENROUTER_MODELS


In [ ]:
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    {"id": "doc:bm25", "text": "BM25 is strong for exact entities, identifiers, numbers, and rare terms."},
    {"id": "doc:dense", "text": "Dense embeddings retrieve passages by semantic similarity even when words differ."},
    {"id": "doc:hybrid", "text": "Hybrid retrieval combines lexical and semantic search before reranking."},
    {"id": "doc:rerank", "text": "A cross encoder or reranker scores query document pairs after initial retrieval."},
]

query = "How do I combine keyword search with semantic retrieval?"
texts = [doc["text"] for doc in documents]
doc_ids = [doc["id"] for doc in documents]

In [ ]:
def tokenize(text):
    return text.lower().replace("?", "").replace(".", "").split()

bm25 = BM25Okapi([tokenize(text) for text in texts])
bm25_scores = bm25.get_scores(tokenize(query))
bm25_ranked = [doc_id for _, doc_id in sorted(zip(bm25_scores, doc_ids), reverse=True)]
bm25_ranked

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
doc_matrix = vectorizer.fit_transform(texts)
query_vector = vectorizer.transform([query])
dense_scores = cosine_similarity(query_vector, doc_matrix).ravel()
dense_ranked = [doc_id for _, doc_id in sorted(zip(dense_scores, doc_ids), reverse=True)]
dense_ranked

In [ ]:
def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, start=1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return [doc_id for doc_id, _ in sorted(scores.items(), key=lambda item: item[1], reverse=True)]

hybrid_ranked = reciprocal_rank_fusion([bm25_ranked, dense_ranked])
hybrid_ranked

In [ ]:
# Optional Cohere reranking after configuring COHERE_API_KEY.
# from langchain_core.documents import Document
# from langchain_cohere import CohereRerank
# reranker = CohereRerank(model="rerank-english-v3.0")
# docs = [Document(page_content=doc["text"], metadata={"id": doc["id"]}) for doc in documents]
# reranked_docs = reranker.compress_documents(docs, query=query)
# [(doc.metadata["id"], doc.metadata.get("relevance_score")) for doc in reranked_docs]